# COMP5318 Assignment 1: Rice Classification

##### Group number: 69
##### Student 1 SID: 540207680
##### Student 2 SID: 530181464

This notebook implements the required COMP5318 Assignment 1 binary classification pipeline on the Rice dataset. It follows the official assessment structure: data preprocessing, baseline evaluation in Part 1, tuned model evaluation in Part 2, and reflection in Part 3.

## Table of Contents

- [1. Data Pre-processing](#1-data-pre-processing)
- [2. Build Classifiers](#2-build-classifiers)
  - [Part 1: Cross-validation without parameter tuning](#part-1-cross-validation-without-parameter-tuning)
  - [Part 2: Cross-validation with parameter tuning](#part-2-cross-validation-with-parameter-tuning)
  - [Part 2: Results](#part-2-results)
- [3. Reflection and Discussion](#3-reflection-and-discussion)


## **1. Data Pre-processing**

In this assignment, `rice-final2.csv` is treated as a supervised binary classification dataset where the last column is the class label and all preceding columns are numeric features.
Missing values are imputed with column means using `SimpleImputer` to retain all training examples with a consistent rule.
Min-max normalization with `MinMaxScaler` maps each feature to `[0, 1]`, which supports fair comparison across classifiers that are sensitive to feature scales.
Labels are converted from `class1`/`class2` to `0`/`1` so all sklearn models and metrics can be applied consistently.
The printed first 10 rows verify that preprocessing has been applied correctly and that numeric values are reported to four decimal places.


In [ ]:
# Import all libraries
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ignore future warnings
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# Load the rice dataset from the assignment folder
data_path = Path("rice-final2.csv")

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path.resolve()}")

riceDataFrame = pd.read_csv(data_path, na_values="?")

if riceDataFrame.shape[1] < 2:
    raise ValueError("The dataset must contain at least one feature column and one class column.")

X = riceDataFrame.iloc[:, :-1].copy()
y = riceDataFrame.iloc[:, -1].copy()

print("Dataset shape:", riceDataFrame.shape)
print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("Class values:", sorted(y.astype(str).str.strip().unique()))


In [ ]:
# Pre-process the dataset in a reusable way so it works with other files of the same format.
def preprocess_dataset(X, y):
    """Fill missing values, normalize features, and encode class labels."""
    if X.empty:
        raise ValueError("Feature matrix is empty.")
    if y.empty:
        raise ValueError("Target vector is empty.")

    # Convert feature columns to numeric so non-numeric placeholders become NaN.
    X_numeric = X.apply(pd.to_numeric, errors="coerce")

    # Guard against columns that contain no usable values at all.
    fully_missing_cols = X_numeric.columns[X_numeric.isna().all()].tolist()
    if fully_missing_cols:
        raise ValueError(f"These feature columns contain only missing or invalid values: {fully_missing_cols}")

    # Replace missing values with the mean of each column.
    imputer = SimpleImputer(strategy="mean")
    X_imputed = imputer.fit_transform(X_numeric)

    # Scale each feature to the [0, 1] range.
    scaler = MinMaxScaler()
    X_processed = scaler.fit_transform(X_imputed)

    # Encode class labels class1 -> 0 and class2 -> 1.
    y_clean = y.astype(str).str.strip()
    label_map = {"class1": 0, "class2": 1}
    y_processed = y_clean.map(label_map)

    if y_processed.isnull().any():
        unknown_labels = sorted(y_clean[y_processed.isnull()].unique())
        raise ValueError(f"Unexpected class labels found: {unknown_labels}")

    return X_processed, y_processed.astype(int).to_numpy(), imputer, scaler

XProcessed, yProcessed, imputer, scaler = preprocess_dataset(X, y)


In [ ]:
# Print the first ten rows of the pre-processed dataset using four decimal places.
def print_data(X, y, n_rows=10):
    """Print the first n_rows of a numpy feature matrix and label vector."""
    if len(X) != len(y):
        raise ValueError("Feature matrix and target vector must have the same number of rows.")

    rows_to_print = min(n_rows, len(X))
    for example_num in range(rows_to_print):
        feature_text = ",".join(f"{feature:.4f}" for feature in X[example_num])
        print(f"{feature_text},{int(y[example_num])}")

print_data(XProcessed, yProcessed)


In [ ]:
# Display the first ten rows as a formatted table for a cleaner notebook preview.
from IPython.display import display

def preview_data_table(X, y, n_rows=10):
    """Display the first n_rows as a styled table without changing the preprocessing logic."""
    if len(X) != len(y):
        raise ValueError("Feature matrix and target vector must have the same number of rows.")

    rows_to_show = min(n_rows, len(X))
    feature_cols = [f"Feature {i + 1}" for i in range(X.shape[1])]
    preview_df = pd.DataFrame(X[:rows_to_show], columns=feature_cols)
    preview_df["Class"] = y[:rows_to_show].astype(int)

    styled = (
        preview_df.style
        .format({col: "{:.4f}" for col in feature_cols})
        .set_caption("Pre-processed Dataset Preview")
        .set_table_styles([
            {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "16px"), ("font-weight", "600"), ("color", "#111827")]},
            {"selector": "th", "props": [("background-color", "#1f2937"), ("color", "white"), ("text-align", "center"), ("padding", "8px 10px")]},
            {"selector": "td", "props": [("text-align", "center"), ("padding", "8px 10px")]},
            {"selector": "table", "props": [("border-collapse", "collapse"), ("border", "1px solid #d1d5db"), ("border-radius", "8px"), ("overflow", "hidden")]},
        ])
    )
    display(styled)

preview_data_table(XProcessed, yProcessed)


## **2. Build Classifiers**

This section evaluates the required classifiers in two stages:
- Part 1: Logistic Regression and Naive Bayes
- Part 2: KNN, Decision Tree, AdaBoost, Gradient Boosting, Random Forest, and SVM


### Part 1: Cross-validation without parameter tuning

Logistic Regression and Gaussian Naive Bayes are evaluated first as baseline classifiers.
Both models use stratified 10-fold cross-validation with `StratifiedKFold(n_splits=10, shuffle=True, random_state=0)` and no hyperparameter tuning, so the comparison is controlled and directly comparable.


In [ ]:
# Configure 10-fold stratified cross-validation for consistent class proportions in each fold.
cvKFold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)


In [ ]:
# Logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

logRModel = LogisticRegression(
    max_iter=1000,
    random_state=0
)

logRScores = cross_val_score(
    estimator=logRModel,
    X=XProcessed,
    y=yProcessed,
    cv=cvKFold,
    scoring="accuracy"
)

logRAvgAccuracy = logRScores.mean()

# Fit a final Logistic Regression model for external test evaluation.
logRFinalModel = LogisticRegression(
    max_iter=1000,
    random_state=0
)
logRFinalModel.fit(XProcessed, yProcessed)


In [ ]:
# Naive Bayes
from sklearn.naive_bayes import GaussianNB

nbModel = GaussianNB()

nbScores = cross_val_score(
    estimator=nbModel,
    X=XProcessed,
    y=yProcessed,
    cv=cvKFold,
    scoring="accuracy"
)

nbAvgAccuracy = nbScores.mean()

# Fit a final Naive Bayes model for external test evaluation.
nbFinalModel = GaussianNB()
nbFinalModel.fit(XProcessed, yProcessed)


### Part 1 Results

The following output reports the average stratified 10-fold cross-validation accuracy for each baseline classifier.


In [ ]:
# Print results for each classifier in part 1 to 4 decimal places here:
print(f"LogR average cross-validation accuracy: {logRAvgAccuracy:.4f}")
print(f"NB average cross-validation accuracy: {nbAvgAccuracy:.4f}")

The Part 1 results indicate that both baseline models perform strongly on the preprocessed Rice data, with Logistic Regression slightly outperforming Naive Bayes in average cross-validation accuracy. This provides a reliable baseline before introducing hyperparameter tuning in Part 2.


### Part 2: Cross-validation with parameter tuning

The remaining classifiers are tuned using `GridSearchCV` with the same stratified 10-fold splitter (`cvKFold`).
A stratified `train_test_split(..., random_state=0)` is used to create a consistent held-out test set for fair comparison of tuned models.


In [ ]:
# Split the data into training and test sets first so the same split can be reused by Decision Tree, AdaBoost, Gradient Boosting, Random Forest, and SVM.
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

xTrain, xTest, yTrain, yTest = train_test_split(
    XProcessed,
    yProcessed,
    test_size=0.2,
    stratify=yProcessed,
    random_state=0
)


#### KNN

KNN predicts a sample from the labels of nearby training points. Tuning `n_neighbors` and distance metric parameter `p` controls locality and distance sensitivity.


In [ ]:
# KNN
# parameters you may consider
k = [1, 3, 5, 7]
p = [1, 2]

# p=1 -> Manhattan distance
# p=2 -> Euclidean distance
knnParameterGrid = {
    "n_neighbors": k,
    "p": p
}

knnModel = KNeighborsClassifier(metric="minkowski")

knnGridSearch = GridSearchCV(
    estimator=knnModel,
    param_grid=knnParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=1
)

knnGridSearch.fit(xTrain, yTrain)

knnBestModel = knnGridSearch.best_estimator_
knnBestK = knnGridSearch.best_params_["n_neighbors"]
knnBestP = knnGridSearch.best_params_["p"]
knnBestCvAccuracy = knnGridSearch.best_score_

knnTestPrediction = knnBestModel.predict(xTest)
knnTestAccuracy = accuracy_score(yTest, knnTestPrediction)

#### Decision Tree

Decision Trees learn feature-based split rules that are easy to interpret. The grid over depth and split/leaf constraints helps balance expressiveness and overfitting risk.


In [ ]:
# Decision Tree
# parameters you may consider
max_depth = [3, 5, 7, 10]
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

decisionTreeParameterGrid = {
    "max_depth": max_depth,
    "min_samples_split": min_samples_split,
    "min_samples_leaf": min_samples_leaf
}

decisionTreeModel = DecisionTreeClassifier(random_state=0)

decisionTreeGridSearch = GridSearchCV(
    estimator=decisionTreeModel,
    param_grid=decisionTreeParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=1
)

decisionTreeGridSearch.fit(xTrain, yTrain)

decisionTreeBestModel = decisionTreeGridSearch.best_estimator_
decisionTreeBestMaxDepth = decisionTreeGridSearch.best_params_["max_depth"]
decisionTreeBestMinSamplesSplit = decisionTreeGridSearch.best_params_["min_samples_split"]
decisionTreeBestMinSamplesLeaf = decisionTreeGridSearch.best_params_["min_samples_leaf"]
decisionTreeBestCvAccuracy = decisionTreeGridSearch.best_score_

decisionTreeTestPrediction = decisionTreeBestModel.predict(xTest)
decisionTreeTestAccuracy = accuracy_score(yTest, decisionTreeTestPrediction)

#### AdaBoost

AdaBoost combines many weak learners by focusing subsequent learners on harder examples. Tuning `n_estimators` and `learning_rate` controls ensemble strength and update size.


In [ ]:
# Ada Boost
# parameters you may consider
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score

adaBoostParameterGrid = {
    "n_estimators": n_estimators,
    "learning_rate": learning_rate
}

adaBoostModel = AdaBoostClassifier(random_state=0)

adaBoostGridSearch = GridSearchCV(
    estimator=adaBoostModel,
    param_grid=adaBoostParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=1
)

adaBoostGridSearch.fit(xTrain, yTrain)

adaBoostTestPrediction = adaBoostGridSearch.best_estimator_.predict(xTest)
adaBoostTestAccuracy = accuracy_score(yTest, adaBoostTestPrediction)

#### Gradient Boosting

Gradient Boosting fits trees sequentially to reduce residual errors. Tuning tree depth, ensemble size, and learning rate manages the bias-variance trade-off.


In [ ]:
# Gradient Boost
# parameters you may consider
max_depth = [1, 3, 5, 7]
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

gradientBoostParameterGrid = {
    "max_depth": max_depth,
    "n_estimators": n_estimators,
    "learning_rate": learning_rate
}

gradientBoostModel = GradientBoostingClassifier(random_state=0)

gradientBoostGridSearch = GridSearchCV(
    estimator=gradientBoostModel,
    param_grid=gradientBoostParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=1
)

gradientBoostGridSearch.fit(xTrain, yTrain)

gradientBoostTestPrediction = gradientBoostGridSearch.best_estimator_.predict(xTest)
gradientBoostTestAccuracy = accuracy_score(yTest, gradientBoostTestPrediction)

#### Random Forest

Random Forest aggregates many randomized trees to improve robustness. Following the assignment, entropy and `max_features='sqrt'` are used, while grid search tunes ensemble size and `max_leaf_nodes`.


In [ ]:
# Random Forest
# You should use RandomForestClassifier from sklearn.ensemble with information gain and max_features set to 'sqrt'.
# parameters you may consider
n_estimators = [10, 30, 60, 100]
max_leaf_nodes = [6, 12]

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

randomForestParameterGrid = {
    "n_estimators": n_estimators,
    "max_leaf_nodes": max_leaf_nodes
}

randomForestModel = RandomForestClassifier(
    criterion="entropy",      # information gain
    max_features="sqrt",
    random_state=0
)

randomForestGridSearch = GridSearchCV(
    estimator=randomForestModel,
    param_grid=randomForestParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=1
)

randomForestGridSearch.fit(xTrain, yTrain)

randomForestTestPrediction = randomForestGridSearch.best_estimator_.predict(xTest)
randomForestTestAccuracy = accuracy_score(yTest, randomForestTestPrediction)
randomForestMacroF1 = f1_score(yTest, randomForestTestPrediction, average="macro")
randomForestWeightedF1 = f1_score(yTest, randomForestTestPrediction, average="weighted")

#### SVM

SVM with an RBF kernel models nonlinear decision boundaries in transformed feature space. Tuning `C` and `gamma` controls margin flexibility and kernel influence.


In [ ]:
# SVM
# parameters you may consider
C = [0.01, 0.1, 1, 5]
gamma = [0.01, 0.1, 1, 10]

# optional
kernel = ["rbf"]

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

svmParameterGrid = {
    "C": C,
    "gamma": gamma,
    "kernel": kernel
}

svmModel = SVC(random_state=0)

svmGridSearch = GridSearchCV(
    estimator=svmModel,
    param_grid=svmParameterGrid,
    cv=cvKFold,
    scoring="accuracy",
    n_jobs=1
)

svmGridSearch.fit(xTrain, yTrain)

svmTestPrediction = svmGridSearch.best_estimator_.predict(xTest)
svmTestAccuracy = accuracy_score(yTest, svmTestPrediction)


### Part 2: Results

The following output reports the best hyperparameters, cross-validation accuracy, and held-out test accuracy for each tuned classifier (with Random Forest macro/weighted F1 scores as required).


In [ ]:
# Report tuned-model results with assignment-required formatting:
# - floating-point metrics to 4 decimal places
# - integer-valued hyperparameters as integers
print(f"KNN best k: {knnGridSearch.best_params_['n_neighbors']}")
print(f"KNN best p: {knnGridSearch.best_params_['p']}")
print(f"KNN cross-validation accuracy: {knnGridSearch.best_score_:.4f}")
print(f"KNN test set accuracy: {knnTestAccuracy:.4f}")
print()

print(f"Decision Tree best max_depth: {decisionTreeGridSearch.best_params_['max_depth']}")
print(f"Decision Tree best min_samples_split: {decisionTreeGridSearch.best_params_['min_samples_split']}")
print(f"Decision Tree best min_samples_leaf: {decisionTreeGridSearch.best_params_['min_samples_leaf']}")
print(f"Decision Tree cross-validation accuracy: {decisionTreeGridSearch.best_score_:.4f}")
print(f"Decision Tree test set accuracy: {decisionTreeTestAccuracy:.4f}")
print()

print(f"AdaBoost best n_estimators: {adaBoostGridSearch.best_params_['n_estimators']}")
print(f"AdaBoost best learning_rate: {adaBoostGridSearch.best_params_['learning_rate']:.4f}")
print(f"AdaBoost cross-validation accuracy: {adaBoostGridSearch.best_score_:.4f}")
print(f"AdaBoost test set accuracy: {adaBoostTestAccuracy:.4f}")
print()

print(f"Gradient Boost best max_depth: {gradientBoostGridSearch.best_params_['max_depth']}")
print(f"Gradient Boost best n_estimators: {gradientBoostGridSearch.best_params_['n_estimators']}")
print(f"Gradient Boost best learning_rate: {gradientBoostGridSearch.best_params_['learning_rate']:.4f}")
print(f"Gradient Boost cross-validation accuracy: {gradientBoostGridSearch.best_score_:.4f}")
print(f"Gradient Boost test set accuracy: {gradientBoostTestAccuracy:.4f}")
print()

print(f"RF best n_estimators: {randomForestGridSearch.best_params_['n_estimators']}")
print(f"RF best max_leaf_nodes: {randomForestGridSearch.best_params_['max_leaf_nodes']}")
print(f"RF cross-validation accuracy: {randomForestGridSearch.best_score_:.4f}")
print(f"RF test set accuracy: {randomForestTestAccuracy:.4f}")
print(f"RF test set macro average F1: {randomForestMacroF1:.4f}")
print(f"RF test set weighted average F1: {randomForestWeightedF1:.4f}")
print()

print(f"SVM best C: {svmGridSearch.best_params_['C']}")
print(f"SVM best gamma: {svmGridSearch.best_params_['gamma']:.4f}")
print(f"SVM best kernel: {svmGridSearch.best_params_['kernel']}")
print(f"SVM cross-validation accuracy: {svmGridSearch.best_score_:.4f}")
print(f"SVM test set accuracy: {svmTestAccuracy:.4f}")


In this comparison, tuned tree-based and boosting methods achieve the strongest test accuracy, while KNN and SVM are slightly lower on the same split. Cross-validation and test-set values are broadly consistent, indicating stable generalization under the selected preprocessing and evaluation protocol.


### Test your code

`test-before.csv` is retained only as a local runnability check before submission.
Final reported assignment results in this notebook are based on `rice-final2.csv`; external-test outputs should be cleared in the polished submission notebook.


In [ ]:
# Load the test dataset to test out your model (positional-alignment-v2)
from pathlib import Path

import pandas as pd
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

testFileName = "test-before.csv"
testPath = Path(testFileName)

if not testPath.exists():
    raise FileNotFoundError(f"External test dataset not found: {testPath.resolve()}")

testDf = pd.read_csv(testPath, na_values="?")
if testDf.shape[1] < 2:
    raise ValueError("External dataset must contain at least one feature column and one class column.")

xExternalRaw = testDf.iloc[:, :-1].copy()
yExternalRaw = testDf.iloc[:, -1].copy()

labelMap = {"class1": 0, "class2": 1}
yExternalClean = yExternalRaw.astype(str).str.strip()
yExternal = yExternalClean.map(labelMap)

if yExternal.isnull().any():
    unknownLabels = sorted(yExternalClean[yExternal.isnull()].unique())
    raise ValueError(f"Unexpected external class labels found: {unknownLabels}")

yExternal = yExternal.astype(int).to_numpy()

trainingFeatureNames = list(X.columns)
externalFeatureNames = list(xExternalRaw.columns)

print("External test dataset loaded successfully.")
print(f"Dataset shape: {testDf.shape}")
print(f"Feature matrix shape: {xExternalRaw.shape}")
print(f"Target vector shape: {yExternal.shape}")
print(f"Training feature names ({len(trainingFeatureNames)}): {trainingFeatureNames}")
print(f"External feature names ({len(externalFeatureNames)}): {externalFeatureNames}")
print("Using positional feature alignment based on repository column order.")

if len(externalFeatureNames) > len(trainingFeatureNames):
    raise ValueError(
        "Positional alignment is impossible because the external set has more features "
        f"({len(externalFeatureNames)}) than the training set ({len(trainingFeatureNames)})."
    )

alignedFeatureSpace = trainingFeatureNames[:len(externalFeatureNames)]
print(f"Final aligned feature space ({len(alignedFeatureSpace)}): {alignedFeatureSpace}")

xTrainAlignedRaw = X.loc[:, alignedFeatureSpace].copy()
xExternalAlignedRaw = xExternalRaw.copy()
xExternalAlignedRaw.columns = alignedFeatureSpace

xTrainAlignedRaw = xTrainAlignedRaw.apply(pd.to_numeric, errors="coerce")
xExternalAlignedRaw = xExternalAlignedRaw.apply(pd.to_numeric, errors="coerce")

fullyMissingTrainColumns = xTrainAlignedRaw.columns[xTrainAlignedRaw.isna().all()].tolist()
if fullyMissingTrainColumns:
    raise ValueError(f"Aligned training columns are fully missing: {fullyMissingTrainColumns}")

fullyMissingExternalColumns = xExternalAlignedRaw.columns[xExternalAlignedRaw.isna().all()].tolist()
if fullyMissingExternalColumns:
    raise ValueError(f"Aligned external columns are fully missing: {fullyMissingExternalColumns}")

# Fit preprocessing on the aligned training representation only.
alignedImputer = SimpleImputer(strategy="mean")
xTrainAlignedImputed = alignedImputer.fit_transform(xTrainAlignedRaw)
xExternalAlignedImputed = alignedImputer.transform(xExternalAlignedRaw)

alignedScaler = MinMaxScaler()
xTrainAlignedScaled = alignedScaler.fit_transform(xTrainAlignedImputed)
xExternalAlignedScaled = alignedScaler.transform(xExternalAlignedImputed)

rfParams = {**randomForestGridSearch.best_params_, "criterion": "entropy", "max_features": "sqrt", "random_state": 0}

# Fit final models on the aligned training representation.
allModels = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=0).fit(xTrainAlignedScaled, yProcessed),
    "Naive Bayes": GaussianNB().fit(xTrainAlignedScaled, yProcessed),
    "KNN": KNeighborsClassifier(metric="minkowski", **knnGridSearch.best_params_).fit(xTrainAlignedScaled, yProcessed),
    "Decision Tree": DecisionTreeClassifier(random_state=0, **decisionTreeGridSearch.best_params_).fit(xTrainAlignedScaled, yProcessed),
    "AdaBoost": AdaBoostClassifier(random_state=0, **adaBoostGridSearch.best_params_).fit(xTrainAlignedScaled, yProcessed),
    "Gradient Boosting": GradientBoostingClassifier(random_state=0, **gradientBoostGridSearch.best_params_).fit(xTrainAlignedScaled, yProcessed),
    "Random Forest": RandomForestClassifier(**rfParams).fit(xTrainAlignedScaled, yProcessed),
    "SVM": SVC(random_state=0, **svmGridSearch.best_params_).fit(xTrainAlignedScaled, yProcessed)
}

print()
print("External test results on test-before.csv")
print("-" * 60)

for modelName, model in allModels.items():
    yExternalPred = model.predict(xExternalAlignedScaled)

    externalAccuracy = accuracy_score(yExternal, yExternalPred)
    externalMacroF1 = f1_score(yExternal, yExternalPred, average="macro")
    externalWeightedF1 = f1_score(yExternal, yExternalPred, average="weighted")

    print(f"{modelName} external test accuracy: {externalAccuracy:.4f}")
    print(f"{modelName} external test macro average F1: {externalMacroF1:.4f}")
    print(f"{modelName} external test weighted average F1: {externalWeightedF1:.4f}")
    print()


## **3. Reflection and Discussion**


Across all eight classifiers, the performance is consistently strong after preprocessing, which suggests that the Rice dataset is relatively well separated in the available feature space. In Part 1, Logistic Regression achieved an average 10-fold cross-validation accuracy of 0.9386, while Gaussian Naive Bayes achieved 0.9264. In Part 2, the best cross-validation scores on the training split were 0.9375 for KNN, 0.9357 for Decision Tree, 0.9437 for AdaBoost, 0.9446 for Gradient Boosting, 0.9411 for Random Forest, and 0.9429 for SVM. On the held-out test set, Decision Tree, Gradient Boosting, and Random Forest all reached 0.9429, AdaBoost reached 0.9393, SVM reached 0.9321, and KNN reached 0.9250. These results indicate that several models perform competitively and that no single complex model dominates the task by a large margin.

The impact of hyperparameter tuning is positive but modest rather than dramatic. The strongest tuned cross-validation result, obtained by Gradient Boosting (0.9446), is only 0.0060 higher than the Logistic Regression baseline in Part 1 (0.9386). This suggests that careful preprocessing and fair evaluation are more important than model complexity alone on this dataset. A useful additional observation is that when Logistic Regression is evaluated on the same held-out train/test split, it achieves a test accuracy of 0.9464, which is slightly higher than the best Part 2 test accuracy of 0.9429. This means that a simple linear model is already highly competitive, so increased model complexity does not automatically translate into better generalization.

The tuning results also show that several best parameters occur at the edges of the search grids, such as KNN with k=7, SVM with C=5, Decision Tree with max_depth=3, AdaBoost with learning_rate=0.1, Gradient Boosting with max_depth=1, n_estimators=50, learning_rate=0.1, and Random Forest with max_leaf_nodes=6. This indicates that the current search space is reasonable but may not fully cover the optimal region. If further tuning were allowed, the grid could be expanded around these boundary values, but model selection should still be based only on cross-validation over the training data rather than on the test set. For Random Forest, the macro F1 score (0.9414) and weighted F1 score (0.9427) are very close, which suggests that performance is balanced across both classes despite the mild class imbalance in the dataset.

From a practical perspective, the current pipeline is reusable because it does not hard-code the number of features and always treats the last column as the class label. However, for a more production-ready workflow, preprocessing should ideally be placed inside a Pipeline so that imputation and scaling are fitted only on the training folds. This is especially important when data come from a database or another external source, where schema drift, missing-value patterns, and type inconsistencies may appear over time. In practice, Logistic Regression is attractive when interpretability, speed, and ease of deployment are priorities, while Random Forest or Gradient Boosting may be preferred when capturing nonlinear interactions is more important. Overall, this assignment shows that strong preprocessing and rigorous evaluation can be as important as sophisticated model choice.